# AdaBoost

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.ensemble import StackingClassifier, AdaBoostClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_validate, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings('ignore')


## Load Data

In [2]:
# 1. LOAD DỮ LIỆU ĐÃ SCALED
data_dir = '../data/'
target_col = 'class'

train_df = pd.read_csv(os.path.join(data_dir, 'train_data.csv'))
valid_df = pd.read_csv(os.path.join(data_dir, 'valid_data.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test_data.csv'))

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_valid = valid_df.drop(columns=[target_col])
y_valid = valid_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]


### Danh sách lưu kết quả để xuất CSV

In [3]:
results_list = []

def evaluate_model(model, name, X_split, y_split, split_name):
    """Dự đoán và trích xuất các chỉ số phân loại theo yêu cầu (Acc, Prec, Rec, F1, AUC)."""
    y_pred = model.predict(X_split)
    y_prob = model.predict_proba(X_split)[:, 1]  # Xác suất class 1 để tính AUC

    acc = accuracy_score(y_split, y_pred)
    prec = precision_score(y_split, y_pred)
    rec = recall_score(y_split, y_pred)
    f1 = f1_score(y_split, y_pred)
    auc = roc_auc_score(y_split, y_prob)

    res = {
        'Algorithm': name,
        'Split': split_name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'AUC': round(auc, 4)
    }

    results_list.append(res)
    print(f"[{name} | {split_name}] Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
    return res


In [4]:
# K-FOLD CONFIG
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_kfold(model, name, X_data, y_data, cv_split):
    """Đánh giá K-Fold và lưu kết quả trung bình vào results_list."""
    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'auc': 'roc_auc'
    }

    cv_results = cross_validate(
        model,
        X_data,
        y_data,
        cv=cv_split,
        scoring=scoring,
        n_jobs=-1
    )

    res = {
        'Algorithm': name,
        'Split': 'KFold',
        'Accuracy': round(cv_results['test_accuracy'].mean(), 4),
        'Precision': round(cv_results['test_precision'].mean(), 4),
        'Recall': round(cv_results['test_recall'].mean(), 4),
        'F1-Score': round(cv_results['test_f1'].mean(), 4),
        'AUC': round(cv_results['test_auc'].mean(), 4)
    }

    results_list.append(res)
    print(
        f"[{name} | KFold] Acc: {res['Accuracy']:.4f} | Prec: {res['Precision']:.4f} | "
        f"Rec: {res['Recall']:.4f} | F1: {res['F1-Score']:.4f} | AUC: {res['AUC']:.4f}"
    )
    return res


### 3. CHẠY BASELINE MODEL (Tham số mặc định)

In [5]:
print("--- V1: Baseline AdaBoost ---")
AdaBoost_baseline_model = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=42), random_state=42)

# Đánh giá K-Fold trước
evaluate_kfold(AdaBoost_baseline_model, "V1: AdaBoost Baseline", X_train, y_train, kfold)

# Train cố định trên train set rồi đánh giá validation/test
AdaBoost_baseline_model.fit(X_train, y_train)
evaluate_model(AdaBoost_baseline_model, "V1: AdaBoost Baseline", X_valid, y_valid, "Validation")
evaluate_model(AdaBoost_baseline_model, "V1: AdaBoost Baseline", X_test, y_test, "Test")


--- V1: Baseline AdaBoost ---
[V1: AdaBoost Baseline | KFold] Acc: 0.9865 | Prec: 0.9797 | Rec: 0.9913 | F1: 0.9854 | AUC: 0.9987
[V1: AdaBoost Baseline | Validation] Acc: 0.9901 | Prec: 0.9891 | Rec: 0.9891 | F1: 0.9891 | AUC: 0.9999
[V1: AdaBoost Baseline | Test] Acc: 0.9975 | Prec: 0.9946 | Rec: 1.0000 | F1: 0.9973 | AUC: 0.9992


{'Algorithm': 'V1: AdaBoost Baseline',
 'Split': 'Test',
 'Accuracy': 0.9975,
 'Precision': 0.9946,
 'Recall': 1.0,
 'F1-Score': 0.9973,
 'AUC': 0.9992}

### 4. CHẠY TUNING MODEL (Dùng GridSearchCV)

In [6]:
print("--- V2: GridSearchCV Tuning ---")
param_grid = {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 1.0], 'estimator__max_depth': [1, 2]}

grid_search = GridSearchCV(
    AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=42), random_state=42),
    param_grid,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
AdaBoost_tuned_model = grid_search.best_estimator_

print(f"Best Params found by CV: {grid_search.best_params_}")

# Đánh giá K-Fold cho mô hình tốt nhất
evaluate_kfold(AdaBoost_tuned_model, "V2: AdaBoost Tuned (GridSearch)", X_train, y_train, kfold)

evaluate_model(AdaBoost_tuned_model, "V2: AdaBoost Tuned (GridSearch)", X_valid, y_valid, "Validation")
evaluate_model(AdaBoost_tuned_model, "V2: AdaBoost Tuned (GridSearch)", X_test, y_test, "Test")


--- V2: GridSearchCV Tuning ---
Best Params found by CV: {'estimator__max_depth': 2, 'learning_rate': 1.0, 'n_estimators': 100}
[V2: AdaBoost Tuned (GridSearch) | KFold] Acc: 0.9986 | Prec: 0.9972 | Rec: 1.0000 | F1: 0.9986 | AUC: 0.9999
[V2: AdaBoost Tuned (GridSearch) | Validation] Acc: 1.0000 | Prec: 1.0000 | Rec: 1.0000 | F1: 1.0000 | AUC: 1.0000
[V2: AdaBoost Tuned (GridSearch) | Test] Acc: 1.0000 | Prec: 1.0000 | Rec: 1.0000 | F1: 1.0000 | AUC: 1.0000


{'Algorithm': 'V2: AdaBoost Tuned (GridSearch)',
 'Split': 'Test',
 'Accuracy': 1.0,
 'Precision': 1.0,
 'Recall': 1.0,
 'F1-Score': 1.0,
 'AUC': 1.0}

In [7]:
print("--- V3: Ensemble Learning (Stacking) ---")
# Kết hợp mô hình chính và KNN làm Base Models
base_estimators = [('ada', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=42), random_state=42)), ('knn', KNeighborsClassifier(n_neighbors=5))]

AdaBoost_stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=DecisionTreeClassifier(random_state=42),
    cv=kfold
)

# Đánh giá K-Fold cho stacking
evaluate_kfold(AdaBoost_stacking_model, "V3: AdaBoost Stacking Ensemble", X_train, y_train, kfold)

AdaBoost_stacking_model.fit(X_train, y_train)
evaluate_model(AdaBoost_stacking_model, "V3: AdaBoost Stacking Ensemble", X_valid, y_valid, "Validation")
evaluate_model(AdaBoost_stacking_model, "V3: AdaBoost Stacking Ensemble", X_test, y_test, "Test")


--- V3: Ensemble Learning (Stacking) ---
[V3: AdaBoost Stacking Ensemble | KFold] Acc: 0.9959 | Prec: 0.9972 | Rec: 0.9943 | F1: 0.9957 | AUC: 0.9959
[V3: AdaBoost Stacking Ensemble | Validation] Acc: 1.0000 | Prec: 1.0000 | Rec: 1.0000 | F1: 1.0000 | AUC: 1.0000
[V3: AdaBoost Stacking Ensemble | Test] Acc: 0.9827 | Prec: 0.9944 | Rec: 0.9672 | F1: 0.9806 | AUC: 0.9814


{'Algorithm': 'V3: AdaBoost Stacking Ensemble',
 'Split': 'Test',
 'Accuracy': 0.9827,
 'Precision': 0.9944,
 'Recall': 0.9672,
 'F1-Score': 0.9806,
 'AUC': 0.9814}

### (5) Lưu kết quả

In [8]:
# CẤU HÌNH ĐƯỜNG DẪN LƯU
save_path = '../experiment/AdaBoost/'
os.makedirs(save_path, exist_ok=True)

# 1. TỔNG HỢP & LƯU 1 FILE CSV DUY NHẤT
df_results = pd.DataFrame(results_list)

csv_output = os.path.join(save_path, 'AdaBoost_full_results.csv')
df_results.to_csv(csv_output, index=False)

# 2. LƯU MODELS (.pkl)
with open(os.path.join(save_path, 'AdaBoost_baseline.pkl'), 'wb') as f:
    pickle.dump(AdaBoost_baseline_model, f)

with open(os.path.join(save_path, 'AdaBoost_tuned.pkl'), 'wb') as f:
    pickle.dump(AdaBoost_tuned_model, f)

with open(os.path.join(save_path, 'AdaBoost_stacking.pkl'), 'wb') as f:
    pickle.dump(AdaBoost_stacking_model, f)

print("-" * 40)
print(f"✅ Đã lưu tất cả kết quả vào: {csv_output}")
print(f"✅ Đã lưu models tại: {save_path}")
print("-" * 40)

display(df_results)


----------------------------------------
✅ Đã lưu tất cả kết quả vào: ../experiment/AdaBoost/AdaBoost_full_results.csv
✅ Đã lưu models tại: ../experiment/AdaBoost/
----------------------------------------


,Algorithm,Split,Accuracy,Precision,Recall,F1-Score,AUC
0,V1: AdaBoost Baseline,KFold,0.9865,0.9797,0.9913,0.9854,0.9987
1,V1: AdaBoost Baseline,Validation,0.9901,0.9891,0.9891,0.9891,0.9999
2,V1: AdaBoost Baseline,Test,0.9975,0.9946,1.0000,0.9973,0.9992
3,V2: AdaBoost Tuned (GridSearch),KFold,0.9986,0.9972,1.0000,0.9986,0.9999
4,V2: AdaBoost Tuned (GridSearch),Validation,1.0000,1.0000,1.0000,1.0000,1.0000
5,V2: AdaBoost Tuned (GridSearch),Test,1.0000,1.0000,1.0000,1.0000,1.0000
6,V3: AdaBoost Stacking Ensemble,KFold,0.9959,0.9972,0.9943,0.9957,0.9959
7,V3: AdaBoost Stacking Ensemble,Validation,1.0000,1.0000,1.0000,1.0000,1.0000
8,V3: AdaBoost Stacking Ensemble,Test,0.9827,0.9944,0.9672,0.9806,0.9814


In [9]:
!jupyter nbconvert --to html AdaBoost_model.ipynb

[NbConvertApp] Converting notebook AdaBoost_model.ipynb to html
[NbConvertApp] Writing 308823 bytes to AdaBoost_model.html
